In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import pickle
import os

# ── Step 15 — Load and clean ──────────────────────────────────────────────
df = pd.read_csv('/Users/vihaan/Desktop/f1-strategy-ai/Data/f1_laps_combined.csv')

cols = ['Driver', 'LapNumber', 'Compound', 'TyreLife', 'LapTime', 'Race', 'Year']
df = df[cols].dropna()

df['LapTime'] = pd.to_timedelta(df['LapTime']).dt.total_seconds()

# Per-circuit bounds (data-driven from actual lap times)
circuit_bounds = {
    'Abu Dhabi Grand Prix':       (85.8, 93.0),
    'Australian Grand Prix':      (78.6, 96.0),
    'Austrian Grand Prix':        (67.1, 75.2),
    'Azerbaijan Grand Prix':      (102.5, 111.6),
    'Bahrain Grand Prix':         (92.6, 102.0),
    'Belgian Grand Prix':         (104.3, 117.9),
    'British Grand Prix':         (87.5, 105.2),
    'Canadian Grand Prix':        (74.2, 105.9),
    'Chinese Grand Prix':         (96.8, 138.0),
    'Dutch Grand Prix':           (72.8, 90.9),
    'Emilia Romagna Grand Prix':  (78.6, 83.5),
    'Hungarian Grand Prix':       (80.2, 87.2),
    'Italian Grand Prix':         (80.6, 89.0),
    'Japanese Grand Prix':        (92.5, 101.9),
    'Las Vegas Grand Prix':       (94.0, 105.6),
    'Mexico City Grand Prix':     (78.6, 87.0),
    'Miami Grand Prix':           (88.9, 96.0),
    'Monaco Grand Prix':          (74.2, 94.8),
    'Qatar Grand Prix':           (81.7, 118.1),
    'Saudi Arabian Grand Prix':   (90.4, 97.6),
    'Singapore Grand Prix':       (94.1, 103.3),
    'Spanish Grand Prix':         (76.0, 83.2),
    'São Paulo Grand Prix':       (72.4, 94.1),
    'United States Grand Prix':   (96.0, 105.1),
}

# Apply per-circuit filter
filtered = []
for race, group in df.groupby('Race'):
    low, high = circuit_bounds.get(race, (60, 180))
    clean = group[(group['LapTime'] > low) & (group['LapTime'] < high)]
    filtered.append(clean)

df = pd.concat(filtered, ignore_index=True)

print("Cleaned data shape:", df.shape)
print("\nAverage lap time per circuit:")
print(df.groupby('Race')['LapTime'].mean().round(2).sort_values())

# ── Step 16 — Encode features ─────────────────────────────────────────────
compound_map = {'SOFT': 0, 'MEDIUM': 1, 'HARD': 2,
                'INTERMEDIATE': 3, 'WET': 4}
df['CompoundCode'] = df['Compound'].map(compound_map)
df = df.dropna(subset=['CompoundCode'])
df['CompoundCode'] = df['CompoundCode'].astype(int)

# One-hot encode race
df = pd.get_dummies(df, columns=['Race'], prefix='Race')

# ── Step 17 — Define features ─────────────────────────────────────────────
base_features = ['LapNumber', 'TyreLife', 'CompoundCode']
race_features = [col for col in df.columns if col.startswith('Race_')]
features = base_features + race_features
target = 'LapTime'

print("\nTotal features:", len(features))
print("Race columns:", race_features)

X = df[features]
y = df[target]

# ── Step 18 — Split ───────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples:  {len(X_test)}")

# ── Step 19 — Train ───────────────────────────────────────────────────────
model = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# ── Step 20 — Evaluate ────────────────────────────────────────────────────
predictions = model.predict(X_test)
mae = mean_absolute_error(y_test, predictions)
r2  = r2_score(y_test, predictions)
print(f"\nMAE: {mae:.2f}s")
print(f"R²:  {r2:.4f}")

# ── Step 21 — Visualise ───────────────────────────────────────────────────
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.2, color='red', s=10)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'w--', lw=1.5)
plt.xlabel('Actual Lap Time (s)')
plt.ylabel('Predicted Lap Time (s)')
plt.title('Predicted vs Actual Lap Times')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Step 22 — Simulate strategy ───────────────────────────────────────────
import json

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'r') as f:
    all_degradation_rates = json.load(f)

def get_deg_rate(race_name, compound):
    global_fallback = {'SOFT': 0.04, 'MEDIUM': 0.025, 'HARD': 0.012}
    return all_degradation_rates.get(race_name, {}).get(
        compound, global_fallback.get(compound, 0.025))

def simulate_strategy(strategy, race_name, total_laps=57):
    total_time = 0
    lap = 1
    lap_times = []

    for compound, stint_laps in strategy:
        compound_code = compound_map.get(compound, 1)
        deg_rate = get_deg_rate(race_name, compound)

        for tyre_life in range(1, stint_laps + 1):
            row = {col: 0 for col in features}
            row['LapNumber']    = lap
            row['TyreLife']     = tyre_life
            row['CompoundCode'] = compound_code
            race_col = f'Race_{race_name}'
            if race_col in row:
                row[race_col] = 1

            input_df = pd.DataFrame([row])[features]
            pred = model.predict(input_df)[0]

            # Apply circuit-specific linear degradation only
            degradation_penalty = deg_rate * tyre_life
            pred += degradation_penalty

            total_time += pred
            lap_times.append({'Lap': lap, 'LapTime': pred,
                              'Compound': compound, 'TyreLife': tyre_life})
            lap += 1

    total_time += (len(strategy) - 1) * 22
    return total_time, pd.DataFrame(lap_times)

# Test it
s1 = simulate_strategy([('SOFT', 15), ('MEDIUM', 22), ('HARD', 20)], 'Bahrain Grand Prix')
s2 = simulate_strategy([('MEDIUM', 30), ('HARD', 27)], 'Bahrain Grand Prix')
print(f"\nStrategy 1 (S→M→H): {s1[0]:.1f}s  ({int(s1[0]//60)}m {s1[0]%60:.1f}s)")
print(f"Strategy 2 (M→H):   {s2[0]:.1f}s  ({int(s2[0]//60)}m {s2[0]%60:.1f}s)")
print(f"Difference: {abs(s1[0]-s2[0]):.1f}s")

# ── Step 23 — Save model ──────────────────────────────────────────────────
save_path    = '/Users/vihaan/Desktop/f1-strategy-ai/Model/f1_strategy_model.pkl'
feature_path = '/Users/vihaan/Desktop/f1-strategy-ai/Model/feature_columns.pkl'

with open(save_path, 'wb') as f:
    pickle.dump(model, f)

with open(feature_path, 'wb') as f:
    pickle.dump(features, f)

print("\nModel saved!")
print("Features saved!")


Race                                 SOFT   MEDIUM     HARD  Ranking
----------------------------------------------------------------------
Abu Dhabi Grand Prix             [0.0350]   0.0254   0.0323       ★★
Australian Grand Prix            [0.0550]   0.0276   0.0388      ★★★
Austrian Grand Prix              [0.0800]   0.0828   0.0747     ★★★★
Azerbaijan Grand Prix            [0.0150]   0.0250   0.0035        ★
Bahrain Grand Prix                 0.0702 [0.0520]   0.0637     ★★★★
Belgian Grand Prix                 0.1146   0.1134   0.0684     ★★★★
British Grand Prix                 0.1112   0.1059   0.0750     ★★★★
Canadian Grand Prix              [0.0350]   0.0125   0.0325       ★★
Chinese Grand Prix               [0.0550]   0.0875   0.0100      ★★★
Dutch Grand Prix                   0.0441   0.0554   0.0409      ★★★
Emilia Romagna Grand Prix        [0.0350]   0.0550   0.0325       ★★
Hungarian Grand Prix             [0.0550]   0.0471   0.0500      ★★★
Italian Grand Prix             

In [7]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import json

df = pd.read_csv('/Users/vihaan/Desktop/f1-strategy-ai/Data/f1_laps_combined.csv')
df['LapTime'] = pd.to_timedelta(df['LapTime']).dt.total_seconds()
df = df.dropna(subset=['LapTime', 'Compound', 'TyreLife', 'Race'])

circuit_bounds = {
    'Abu Dhabi Grand Prix':       (85.8, 93.0),
    'Australian Grand Prix':      (78.6, 96.0),
    'Austrian Grand Prix':        (67.1, 75.2),
    'Azerbaijan Grand Prix':      (102.5, 111.6),
    'Bahrain Grand Prix':         (92.6, 102.0),
    'Belgian Grand Prix':         (104.3, 117.9),
    'British Grand Prix':         (87.5, 105.2),
    'Canadian Grand Prix':        (74.2, 105.9),
    'Chinese Grand Prix':         (96.8, 138.0),
    'Dutch Grand Prix':           (72.8, 90.9),
    'Emilia Romagna Grand Prix':  (78.6, 83.5),
    'Hungarian Grand Prix':       (80.2, 87.2),
    'Italian Grand Prix':         (80.6, 89.0),
    'Japanese Grand Prix':        (92.5, 101.9),
    'Las Vegas Grand Prix':       (94.0, 105.6),
    'Mexico City Grand Prix':     (78.6, 87.0),
    'Miami Grand Prix':           (88.9, 96.0),
    'Monaco Grand Prix':          (74.2, 94.8),
    'Qatar Grand Prix':           (81.7, 118.1),
    'Saudi Arabian Grand Prix':   (90.4, 97.6),
    'Singapore Grand Prix':       (94.1, 103.3),
    'Spanish Grand Prix':         (76.0, 83.2),
    'São Paulo Grand Prix':       (72.4, 94.1),
    'United States Grand Prix':   (96.0, 105.1),
}

# Clean data
filtered = []
for race, group in df.groupby('Race'):
    low, high = circuit_bounds.get(race, (60, 180))
    clean = group[(group['LapTime'] > low) & (group['LapTime'] < high)]
    filtered.append(clean)
df_clean = pd.concat(filtered, ignore_index=True)

# ── Fuel effect correction ────────────────────────────────────────────────
# F1 cars burn approximately 1.8kg/lap
# Each 10kg of fuel = ~0.3s of lap time
# So fuel effect = (1.8 / 10) * 0.3 = 0.054s per lap improvement
# i.e. cars get 0.054s faster each lap purely from fuel burn
FUEL_CORRECTION = 0.054  # seconds per lap

compounds = ['SOFT', 'MEDIUM', 'HARD']

# Known Pirelli degradation rankings by circuit (from Pirelli reports)
# Scale: 1=very low, 2=low, 3=medium, 4=high, 5=very high
pirelli_ranking = {
    'Monaco Grand Prix':          1,
    'Azerbaijan Grand Prix':      1,
    'Saudi Arabian Grand Prix':   1,
    'Las Vegas Grand Prix':       1,
    'Abu Dhabi Grand Prix':       2,
    'Canadian Grand Prix':        2,
    'Mexico City Grand Prix':     2,
    'Emilia Romagna Grand Prix':  2,
    'Italian Grand Prix':         2,
    'Chinese Grand Prix':         3,
    'Miami Grand Prix':           3,
    'Singapore Grand Prix':       3,
    'Australian Grand Prix':      3,
    'Hungarian Grand Prix':       3,
    'Japanese Grand Prix':        3,
    'Dutch Grand Prix':           3,
    'São Paulo Grand Prix':       3,
    'United States Grand Prix':   4,
    'Bahrain Grand Prix':         4,
    'Austrian Grand Prix':        4,
    'Spanish Grand Prix':         4,
    'British Grand Prix':         4,
    'Belgian Grand Prix':         4,
    'Qatar Grand Prix':           5,
}

# Base degradation rates per Pirelli ranking level
# These are the baseline s/lap values before circuit-specific adjustment
base_rates = {
    1: {'SOFT': 0.015, 'MEDIUM': 0.010, 'HARD': 0.007},
    2: {'SOFT': 0.035, 'MEDIUM': 0.022, 'HARD': 0.013},
    3: {'SOFT': 0.055, 'MEDIUM': 0.035, 'HARD': 0.020},
    4: {'SOFT': 0.080, 'MEDIUM': 0.052, 'HARD': 0.030},
    5: {'SOFT': 0.115, 'MEDIUM': 0.078, 'HARD': 0.048},
}

deg_export = {}
print(f"\n{'Race':<32} {'SOFT':>8} {'MEDIUM':>8} {'HARD':>8} {'Ranking':>8}")
print("-" * 70)

for race in sorted(df_clean['Race'].unique()):
    race_data = df_clean[df_clean['Race'] == race]
    ranking = pirelli_ranking.get(race, 3)
    base = base_rates[ranking]
    deg_export[race] = {}
    row_vals = []

    for compound in compounds:
        compound_data = race_data[
            (race_data['Compound'] == compound) &
            (race_data['TyreLife'] > 3)
        ]

        if len(compound_data) < 15:
            # Not enough data — use Pirelli ranking base rate
            final_rate = base[compound]
            deg_export[race][compound] = final_rate
            row_vals.append(f"[{final_rate:.4f}]")
            continue

        # Calculate raw degradation using linear regression
        X = compound_data['TyreLife'].values.reshape(-1, 1)
        y = compound_data['LapTime'].values
        reg = LinearRegression().fit(X, y)
        raw_rate = reg.coef_[0]

        # Apply fuel correction — remove the fuel improvement effect
        # Raw rate is negative when fuel effect dominates
        # True deg = raw rate + fuel correction per lap
        corrected_rate = raw_rate + FUEL_CORRECTION

        # Blend with Pirelli ranking:
        # If we have good data (corrected_rate > 0), weight it 60% data / 40% Pirelli
        # If corrected rate is still negative (very fuel-dominated), use 20% data / 80% Pirelli
        if corrected_rate > 0:
            blended = (0.60 * corrected_rate) + (0.40 * base[compound])
        else:
            blended = (0.20 * corrected_rate) + (0.80 * base[compound])

        # Clamp to realistic range based on Pirelli ranking
        min_rate = base[compound] * 0.5
        max_rate = base[compound] * 2.5
        final_rate = round(max(min_rate, min(blended, max_rate)), 4)

        deg_export[race][compound] = final_rate
        row_vals.append(f"{final_rate:.4f}")

    print(f"{race:<32} {row_vals[0]:>8} {row_vals[1]:>8} {row_vals[2]:>8} {'★' * ranking:>8}")

# Save
with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'w') as f:
    json.dump(deg_export, f, indent=2)

print("\nSaved degradation_rates.json")
print("\nQatar (should be highest):")
print(json.dumps(deg_export.get('Qatar Grand Prix', {}), indent=2))
print("\nMonaco (should be lowest):")
print(json.dumps(deg_export.get('Monaco Grand Prix', {}), indent=2))


Race                                 SOFT   MEDIUM     HARD  Ranking
----------------------------------------------------------------------
Abu Dhabi Grand Prix             [0.0350]   0.0254   0.0323       ★★
Australian Grand Prix            [0.0550]   0.0276   0.0388      ★★★
Austrian Grand Prix              [0.0800]   0.0828   0.0747     ★★★★
Azerbaijan Grand Prix            [0.0150]   0.0250   0.0035        ★
Bahrain Grand Prix                 0.0702 [0.0520]   0.0637     ★★★★
Belgian Grand Prix                 0.1146   0.1134   0.0684     ★★★★
British Grand Prix                 0.1112   0.1059   0.0750     ★★★★
Canadian Grand Prix              [0.0350]   0.0125   0.0325       ★★
Chinese Grand Prix               [0.0550]   0.0875   0.0100      ★★★
Dutch Grand Prix                   0.0441   0.0554   0.0409      ★★★
Emilia Romagna Grand Prix        [0.0350]   0.0550   0.0325       ★★
Hungarian Grand Prix             [0.0550]   0.0471   0.0500      ★★★
Italian Grand Prix             

In [8]:
import json

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'r') as f:
    deg_rates = json.load(f)

# Targeted fixes for physically unrealistic values
fixes = {
    'Qatar Grand Prix': {
        'SOFT':   0.115,   # keep
        'MEDIUM': 0.082,   # Qatar 2023 destroyed mediums in ~15 laps
        'HARD':   0.055,   # hard also degraded heavily at Qatar
    },
    'Azerbaijan Grand Prix': {
        'SOFT':   0.015,   # keep
        'MEDIUM': 0.012,   # keep
        'HARD':   0.008,   # floor — street circuit but not zero
    },
    'Saudi Arabian Grand Prix': {
        'SOFT':   0.015,   # keep
        'MEDIUM': 0.011,   # slight increase from 0.005 floor
        'HARD':   0.008,   # slight increase from 0.005 floor
    },
    'Chinese Grand Prix': {
        'SOFT':   0.055,   # keep
        'MEDIUM': 0.0875,  # keep
        'HARD':   0.022,   # increase from 0.010
    },
    'Miami Grand Prix': {
        'SOFT':   0.055,   # keep
        'MEDIUM': 0.038,   # increase from 0.0175
        'HARD':   0.024,   # slight increase
    },
}

for race, compounds in fixes.items():
    deg_rates[race] = compounds
    print(f"Fixed: {race}")

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'w') as f:
    json.dump(deg_rates, f, indent=2)

print("\nFinal degradation rates:")
for race in sorted(deg_rates.keys()):
    r = deg_rates[race]
    print(f"{race:<32} SOFT:{r['SOFT']:.4f}  MED:{r['MEDIUM']:.4f}  HARD:{r['HARD']:.4f}")

Fixed: Qatar Grand Prix
Fixed: Azerbaijan Grand Prix
Fixed: Saudi Arabian Grand Prix
Fixed: Chinese Grand Prix
Fixed: Miami Grand Prix

Final degradation rates:
Abu Dhabi Grand Prix             SOFT:0.0350  MED:0.0254  HARD:0.0323
Australian Grand Prix            SOFT:0.0550  MED:0.0276  HARD:0.0388
Austrian Grand Prix              SOFT:0.0800  MED:0.0828  HARD:0.0747
Azerbaijan Grand Prix            SOFT:0.0150  MED:0.0120  HARD:0.0080
Bahrain Grand Prix               SOFT:0.0702  MED:0.0520  HARD:0.0637
Belgian Grand Prix               SOFT:0.1146  MED:0.1134  HARD:0.0684
British Grand Prix               SOFT:0.1112  MED:0.1059  HARD:0.0750
Canadian Grand Prix              SOFT:0.0350  MED:0.0125  HARD:0.0325
Chinese Grand Prix               SOFT:0.0550  MED:0.0875  HARD:0.0220
Dutch Grand Prix                 SOFT:0.0441  MED:0.0554  HARD:0.0409
Emilia Romagna Grand Prix        SOFT:0.0350  MED:0.0550  HARD:0.0325
Hungarian Grand Prix             SOFT:0.0550  MED:0.0471  HARD:0.0500

In [9]:
import json

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/circuit_strategies.json', 'r') as f:
    strats = json.load(f)

print(json.dumps(strats.get('Monaco Grand Prix', {}), indent=2))

{
  "typical_stops": 1,
  "compounds_used": [
    "HARD",
    "MEDIUM"
  ],
  "avg_stint_lengths": {
    "HARD": 61.8,
    "MEDIUM": 16.2
  },
  "avg_lap_times": {
    "HARD": 557.58,
    "MEDIUM": 1984.89
  }
}


In [10]:
import json

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'r') as f:
    deg_rates = json.load(f)

# Qatar 2023 — Pirelli mandated 3 stops, tyres were destroyed in 15-18 laps
deg_rates['Qatar Grand Prix'] = {
    'SOFT':   0.200,   # barely usable beyond 10 laps
    'MEDIUM': 0.150,   # destroyed beyond 18-20 laps
    'HARD':   0.095,   # still degraded heavily
}

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/degradation_rates.json', 'w') as f:
    json.dump(deg_rates, f, indent=2)

print("Qatar rates updated:")
print(json.dumps(deg_rates['Qatar Grand Prix'], indent=2))

Qatar rates updated:
{
  "SOFT": 0.2,
  "MEDIUM": 0.15,
  "HARD": 0.095
}


In [14]:
import fastf1
import pandas as pd
import json
import os
import numpy as np

os.makedirs('Data/cache', exist_ok=True)
fastf1.Cache.enable_cache('Data/cache')

# Load multiple years for better statistical accuracy
races_to_load = [
    # 2024
    (2024, 'Bahrain Grand Prix'),
    (2024, 'Saudi Arabian Grand Prix'),
    (2024, 'Australian Grand Prix'),
    (2024, 'Japanese Grand Prix'),
    (2024, 'Chinese Grand Prix'),
    (2024, 'Miami Grand Prix'),
    (2024, 'Emilia Romagna Grand Prix'),
    (2024, 'Monaco Grand Prix'),
    (2024, 'Canadian Grand Prix'),
    (2024, 'Spanish Grand Prix'),
    (2024, 'Austrian Grand Prix'),
    (2024, 'British Grand Prix'),
    (2024, 'Hungarian Grand Prix'),
    (2024, 'Belgian Grand Prix'),
    (2024, 'Dutch Grand Prix'),
    (2024, 'Italian Grand Prix'),
    (2024, 'Azerbaijan Grand Prix'),
    (2024, 'Singapore Grand Prix'),
    (2024, 'United States Grand Prix'),
    (2024, 'Mexico City Grand Prix'),
    (2024, 'São Paulo Grand Prix'),
    (2024, 'Las Vegas Grand Prix'),
    (2024, 'Qatar Grand Prix'),
    (2024, 'Abu Dhabi Grand Prix'),
    # 2023
    (2023, 'Bahrain Grand Prix'),
    (2023, 'Saudi Arabian Grand Prix'),
    (2023, 'Australian Grand Prix'),
    (2023, 'Azerbaijan Grand Prix'),
    (2023, 'Miami Grand Prix'),
    (2023, 'Monaco Grand Prix'),
    (2023, 'Spanish Grand Prix'),
    (2023, 'Canadian Grand Prix'),
    (2023, 'Austrian Grand Prix'),
    (2023, 'British Grand Prix'),
    (2023, 'Hungarian Grand Prix'),
    (2023, 'Belgian Grand Prix'),
    (2023, 'Dutch Grand Prix'),
    (2023, 'Italian Grand Prix'),
    (2023, 'Singapore Grand Prix'),
    (2023, 'Japanese Grand Prix'),
    (2023, 'Qatar Grand Prix'),
    (2023, 'United States Grand Prix'),
    (2023, 'Mexico City Grand Prix'),
    (2023, 'São Paulo Grand Prix'),
    (2023, 'Las Vegas Grand Prix'),
    (2023, 'Abu Dhabi Grand Prix'),
    # 2022 — more data = better probability estimates
    (2022, 'Bahrain Grand Prix'),
    (2022, 'Saudi Arabian Grand Prix'),
    (2022, 'Australian Grand Prix'),
    (2022, 'Emilia Romagna Grand Prix'),
    (2022, 'Miami Grand Prix'),
    (2022, 'Spanish Grand Prix'),
    (2022, 'Monaco Grand Prix'),
    (2022, 'Azerbaijan Grand Prix'),
    (2022, 'Canadian Grand Prix'),
    (2022, 'British Grand Prix'),
    (2022, 'Austrian Grand Prix'),
    (2022, 'French Grand Prix'),
    (2022, 'Hungarian Grand Prix'),
    (2022, 'Belgian Grand Prix'),
    (2022, 'Dutch Grand Prix'),
    (2022, 'Italian Grand Prix'),
    (2022, 'Singapore Grand Prix'),
    (2022, 'Japanese Grand Prix'),
    (2022, 'United States Grand Prix'),
    (2022, 'Mexico City Grand Prix'),
    (2022, 'São Paulo Grand Prix'),
    (2022, 'Abu Dhabi Grand Prix'),
]

# Track characteristics that influence SC probability
# These are fixed physical characteristics of each circuit
track_characteristics = {
    'Monaco Grand Prix':          {'wall_proximity': 1.0, 'overtaking_difficulty': 1.0, 'track_width': 0.3},
    'Singapore Grand Prix':       {'wall_proximity': 0.9, 'overtaking_difficulty': 0.9, 'track_width': 0.4},
    'São Paulo Grand Prix':       {'wall_proximity': 0.6, 'overtaking_difficulty': 0.5, 'track_width': 0.6},
    'Australian Grand Prix':      {'wall_proximity': 0.6, 'overtaking_difficulty': 0.6, 'track_width': 0.6},
    'Azerbaijan Grand Prix':      {'wall_proximity': 0.8, 'overtaking_difficulty': 0.5, 'track_width': 0.5},
    'Saudi Arabian Grand Prix':   {'wall_proximity': 0.8, 'overtaking_difficulty': 0.5, 'track_width': 0.5},
    'Canadian Grand Prix':        {'wall_proximity': 0.7, 'overtaking_difficulty': 0.5, 'track_width': 0.6},
    'Las Vegas Grand Prix':       {'wall_proximity': 0.7, 'overtaking_difficulty': 0.4, 'track_width': 0.6},
    'Miami Grand Prix':           {'wall_proximity': 0.5, 'overtaking_difficulty': 0.5, 'track_width': 0.6},
    'Qatar Grand Prix':           {'wall_proximity': 0.3, 'overtaking_difficulty': 0.4, 'track_width': 0.7},
    'Bahrain Grand Prix':         {'wall_proximity': 0.3, 'overtaking_difficulty': 0.3, 'track_width': 0.8},
    'Belgian Grand Prix':         {'wall_proximity': 0.4, 'overtaking_difficulty': 0.4, 'track_width': 0.7},
    'British Grand Prix':         {'wall_proximity': 0.3, 'overtaking_difficulty': 0.4, 'track_width': 0.8},
    'Austrian Grand Prix':        {'wall_proximity': 0.2, 'overtaking_difficulty': 0.3, 'track_width': 0.9},
    'Dutch Grand Prix':           {'wall_proximity': 0.4, 'overtaking_difficulty': 0.6, 'track_width': 0.6},
    'Hungarian Grand Prix':       {'wall_proximity': 0.3, 'overtaking_difficulty': 0.7, 'track_width': 0.7},
    'Spanish Grand Prix':         {'wall_proximity': 0.2, 'overtaking_difficulty': 0.5, 'track_width': 0.8},
    'Italian Grand Prix':         {'wall_proximity': 0.3, 'overtaking_difficulty': 0.3, 'track_width': 0.9},
    'Japanese Grand Prix':        {'wall_proximity': 0.5, 'overtaking_difficulty': 0.5, 'track_width': 0.7},
    'Chinese Grand Prix':         {'wall_proximity': 0.3, 'overtaking_difficulty': 0.4, 'track_width': 0.8},
    'United States Grand Prix':   {'wall_proximity': 0.3, 'overtaking_difficulty': 0.4, 'track_width': 0.8},
    'Mexico City Grand Prix':     {'wall_proximity': 0.4, 'overtaking_difficulty': 0.5, 'track_width': 0.7},
    'Abu Dhabi Grand Prix':       {'wall_proximity': 0.3, 'overtaking_difficulty': 0.4, 'track_width': 0.8},
    'Emilia Romagna Grand Prix':  {'wall_proximity': 0.5, 'overtaking_difficulty': 0.6, 'track_width': 0.6},
    'Las Vegas Grand Prix':       {'wall_proximity': 0.7, 'overtaking_difficulty': 0.4, 'track_width': 0.6},
}

# Collect raw SC data per circuit
sc_raw = {}

for year, race in races_to_load:
    try:
        session = fastf1.get_session(year, race, 'R')
        session.load(laps=True, telemetry=False, weather=False)
        race_name = session.event['EventName']

        total_laps = session.laps['LapNumber'].max()
        if pd.isna(total_laps) or total_laps == 0:
            continue

        # Count SC status periods
        track_status = session.track_status
        sc_laps = len(track_status[
            track_status['Status'].isin(['4', '5'])  # 4=SC, 5=SC ending
        ])
        vsc_laps = len(track_status[
            track_status['Status'].isin(['6', '7'])  # 6=VSC, 7=VSC ending
        ])

        sc_occurred  = 1 if sc_laps > 0 else 0
        vsc_occurred = 1 if vsc_laps > 0 else 0
        any_sc       = 1 if (sc_laps + vsc_laps) > 0 else 0

        # SC laps as percentage of total race laps
        sc_pct = (sc_laps + vsc_laps * 0.5) / total_laps  # VSC weighted half

        if race_name not in sc_raw:
            sc_raw[race_name] = {
                'sc_races':    0,
                'total_races': 0,
                'sc_laps':     [],
                'sc_pct':      [],
            }

        sc_raw[race_name]['total_races']  += 1
        sc_raw[race_name]['sc_races']     += any_sc
        sc_raw[race_name]['sc_laps'].append(sc_laps + vsc_laps)
        sc_raw[race_name]['sc_pct'].append(sc_pct)

        print(f"✅ {year} {race_name}: SC={sc_laps} VSC={vsc_laps} any={any_sc}")

    except Exception as e:
        print(f"❌ {year} {race}: {e}")

# ── Calculate blended SC probability ─────────────────────────────────────
# Blend: 70% historical data + 30% track characteristics
sc_export = {}

for race, data in sc_raw.items():
    if data['total_races'] == 0:
        continue

    # Historical probability (with Laplace smoothing to avoid 0% or 100%)
    # Laplace smoothing adds 1 success and 1 failure to avoid extremes
    smoothed_prob = (data['sc_races'] + 1) / (data['total_races'] + 2)

    # Average SC laps when SC occurs
    avg_sc_laps = round(np.mean(data['sc_laps']), 1)

    # Track characteristic score (0-1)
    char = track_characteristics.get(race, {
        'wall_proximity': 0.4,
        'overtaking_difficulty': 0.4,
        'track_width': 0.7
    })
    track_score = (
        char['wall_proximity'] * 0.5 +
        char['overtaking_difficulty'] * 0.3 +
        (1 - char['track_width']) * 0.2
    )

    # Blend historical + track characteristics
    blended_prob = (0.70 * smoothed_prob) + (0.30 * track_score)

    # Clamp to 5%-95% — nothing is certain in F1
    blended_prob = round(max(0.05, min(0.95, blended_prob)), 3)

    # Risk level thresholds
    if blended_prob >= 0.70:
        risk_level = 'VERY HIGH'
    elif blended_prob >= 0.50:
        risk_level = 'HIGH'
    elif blended_prob >= 0.30:
        risk_level = 'MEDIUM'
    else:
        risk_level = 'LOW'

    sc_export[race] = {
        'sc_probability':   blended_prob,
        'historical_prob':  round(smoothed_prob, 3),
        'track_score':      round(track_score, 3),
        'avg_sc_laps':      avg_sc_laps,
        'risk_level':       risk_level,
        'races_analysed':   data['total_races'],
        'sc_occurrences':   data['sc_races'],
    }

    print(f"{race:<35} "
          f"hist={smoothed_prob:.1%} "
          f"track={track_score:.1%} "
          f"blended={blended_prob:.1%} "
          f"→ {risk_level}")

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/sc_probability.json', 'w') as f:
    json.dump(sc_export, f, indent=2)

print("\nSaved sc_probability.json")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached dat

✅ 2024 Bahrain Grand Prix: SC=0 VSC=0 any=0


req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✅ 2024 Saudi Arabian Grand Prix: SC=1 VSC=0 any=1


req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

✅ 2022 Mexico City Grand Prix: SC=0 VSC=2 any=1


_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for race_control_messages. Loading data...
_api           INFO 	Fetching race control messages...
req            INFO 	Data has been written to cache!
core           INFO 	Finished loading data for 20 drivers: ['63', '44', '55', '16', '14', '1', '11', '31', '77', '18', '5', '24', '47', '10', '23', '6', '22', '4', '20', '3']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.2]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...


✅ 2022 São Paulo Grand Prix: SC=2 VSC=1 any=1


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

✅ 2022 Abu Dhabi Grand Prix: SC=0 VSC=0 any=0
Bahrain Grand Prix                  hist=60.0% track=28.0% blended=50.4% → HIGH
Saudi Arabian Grand Prix            hist=80.0% track=65.0% blended=75.5% → VERY HIGH
Australian Grand Prix               hist=80.0% track=56.0% blended=72.8% → VERY HIGH
Japanese Grand Prix                 hist=80.0% track=46.0% blended=69.8% → HIGH
Chinese Grand Prix                  hist=66.7% track=31.0% blended=56.0% → HIGH
Miami Grand Prix                    hist=60.0% track=48.0% blended=56.4% → HIGH
Emilia Romagna Grand Prix           hist=50.0% track=51.0% blended=50.3% → HIGH
Monaco Grand Prix                   hist=60.0% track=94.0% blended=70.2% → VERY HIGH
Canadian Grand Prix                 hist=80.0% track=58.0% blended=73.4% → VERY HIGH
Spanish Grand Prix                  hist=20.0% track=29.0% blended=22.7% → LOW
Austrian Grand Prix                 hist=80.0% track=21.0% blended=62.3% → HIGH
British Grand Prix                  hist=60.0% track=31

In [12]:
import fastf1
import pandas as pd
import json
import os

os.makedirs('Data/cache', exist_ok=True)
fastf1.Cache.enable_cache('Data/cache')

races_to_load = [
    (2024, 'Bahrain Grand Prix'),
    (2024, 'Saudi Arabian Grand Prix'),
    (2024, 'Australian Grand Prix'),
    (2024, 'Japanese Grand Prix'),
    (2024, 'Chinese Grand Prix'),
    (2024, 'Miami Grand Prix'),
    (2024, 'Emilia Romagna Grand Prix'),
    (2024, 'Monaco Grand Prix'),
    (2024, 'Canadian Grand Prix'),
    (2024, 'Spanish Grand Prix'),
    (2024, 'Austrian Grand Prix'),
    (2024, 'British Grand Prix'),
    (2024, 'Hungarian Grand Prix'),
    (2024, 'Belgian Grand Prix'),
    (2024, 'Dutch Grand Prix'),
    (2024, 'Italian Grand Prix'),
    (2024, 'Azerbaijan Grand Prix'),
    (2024, 'Singapore Grand Prix'),
    (2024, 'United States Grand Prix'),
    (2024, 'Mexico City Grand Prix'),
    (2024, 'São Paulo Grand Prix'),
    (2024, 'Las Vegas Grand Prix'),
    (2024, 'Qatar Grand Prix'),
    (2024, 'Abu Dhabi Grand Prix'),
]

winner_strategies = {}

for year, race in races_to_load:
    try:
        session = fastf1.get_session(year, race, 'R')
        session.load()
        race_name = session.event['EventName']

        # Get race winner
        winner = session.results.head(1)['Abbreviation'].iloc[0]
        winner_full = session.results.head(1)['FullName'].iloc[0]

        # Get winner laps
        laps = session.laps.pick_drivers(winner).copy()
        laps['LapTimeSeconds'] = laps['LapTime'].dt.total_seconds()

        # Extract stints
        stints = (
            laps.groupby(['Stint', 'Compound'])
            .agg(
                StintLength  =('LapNumber', 'count'),
                AvgLapTime   =('LapTimeSeconds', 'mean'),
                MinLapTime   =('LapTimeSeconds', 'min'),
            )
            .reset_index()
            .sort_values('Stint')
        )

        # Build strategy list
        strategy_list = []
        for _, row in stints.iterrows():
            if pd.notna(row['Compound']) and row['StintLength'] > 0:
                strategy_list.append({
                    'compound':    row['Compound'],
                    'laps':        int(row['StintLength']),
                    'avg_lap':     round(row['AvgLapTime'], 3),
                    'fastest_lap': round(row['MinLapTime'], 3),
                })

        # Total race time
        total_time = laps['LapTimeSeconds'].sum()
        pit_stops  = len(strategy_list) - 1

        winner_strategies[race_name] = {
            'year':         year,
            'winner':       winner,
            'winner_full':  winner_full,
            'strategy':     strategy_list,
            'total_laps':   int(laps['LapNumber'].max()),
            'pit_stops':    pit_stops,
        }

        strategy_str = ' → '.join(
            [f"{s['compound']} ({s['laps']})" for s in strategy_list])
        print(f"✅ {year} {race_name}: {winner_full} — {strategy_str}")

    except Exception as e:
        print(f"❌ {year} {race}: {e}")

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/winner_strategies.json', 'w') as f:
    json.dump(winner_strategies, f, indent=2)

print("\nSaved winner_strategies.json")

core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '63', '4', '44', '81', '14', '18', '24', '20', '3', '22', '23', '27', '31', '10', '77', '2']
core           INFO 	Loading data for Saudi Arabian Grand Prix

✅ 2024 Bahrain Grand Prix: Max Verstappen — SOFT (17) → HARD (20) → SOFT (20)


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '16', '81', '14', '63', '38', '4', '44', '27', '23', '20', '31', '2', '22', '3', '77', '24', '18', '10']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


✅ 2024 Saudi Arabian Grand Prix: Max Verstappen — MEDIUM (7) → HARD (43)


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 19 drivers: ['55', '16', '4', '81', '11', '18', '22', '14', '27', '20', '23', '3', '10', '77', '24', '31', '63', '44', '1']
core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

✅ 2024 Las Vegas Grand Prix: George Russell — MEDIUM (12) → HARD (20) → HARD (18)


core        WARNING 	Fixed incorrect tyre stint information for driver '43'
core        WARNING 	Fixed incorrect tyre stint information for driver '31'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '81', '63', '10', '55', '14', '24', '20', '4', '77', '44', '22', '30', '23', '27', '11', '18', '43', '31']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.2]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req         

✅ 2024 Qatar Grand Prix: Max Verstappen — MEDIUM (35) → HARD (1) → HARD (1) → HARD (20)


req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '55', '16', '44', '63', '1', '10', '27', '14', '81', '23', '22', '24', '18', '61', '20', '30', '77', '43', '11']


✅ 2024 Abu Dhabi Grand Prix: Lando Norris — MEDIUM (26) → HARD (32)

Saved winner_strategies.json


In [15]:
import json
import numpy as np

# ── Fuel load calculator ──────────────────────────────────────────────────
# F1 fuel facts (from Pirelli/FIA technical regulations):
# - Maximum fuel load: 110kg
# - Fuel burn rate: approximately 1.8kg per racing lap on average
#   (varies by circuit — faster circuits burn more, street circuits less)
# - Every 10kg of fuel = approximately 0.32 seconds per lap (FIA estimated)
# - Therefore: fuel effect per lap = 1.8 * (0.32/10) = 0.0576s per lap improvement
#   as fuel burns off

# Circuit-specific fuel consumption rates (kg per lap)
# Based on circuit length, speed profile, and engine load
# Sources: Pirelli/Mercedes/Red Bull technical briefings and F1 technical regulations
fuel_consumption_rates = {
    'Abu Dhabi Grand Prix':      1.75,
    'Australian Grand Prix':     1.65,
    'Austrian Grand Prix':       1.50,
    'Azerbaijan Grand Prix':     1.70,
    'Bahrain Grand Prix':        1.80,
    'Belgian Grand Prix':        2.10,  # Spa — longest circuit, highest consumption
    'British Grand Prix':        1.90,
    'Canadian Grand Prix':       1.70,
    'Chinese Grand Prix':        1.80,
    'Dutch Grand Prix':          1.55,
    'Emilia Romagna Grand Prix': 1.65,
    'Hungarian Grand Prix':      1.55,
    'Italian Grand Prix':        1.95,  # Monza — high speed, high fuel burn
    'Japanese Grand Prix':       1.85,
    'Las Vegas Grand Prix':      1.75,
    'Mexico City Grand Prix':    1.50,  # High altitude reduces fuel consumption
    'Miami Grand Prix':          1.70,
    'Monaco Grand Prix':         1.45,  # Slow circuit, lowest consumption
    'Qatar Grand Prix':          1.80,
    'Saudi Arabian Grand Prix':  1.85,
    'Singapore Grand Prix':      1.55,  # Street circuit, low speed
    'Spanish Grand Prix':        1.75,
    'São Paulo Grand Prix':      1.70,
    'United States Grand Prix':  1.80,
}

# Fuel time effect: seconds per lap per kg of fuel
# FIA regulation estimate: 0.032s per lap per kg
FUEL_TIME_EFFECT = 0.032  # s/lap/kg

fuel_export = {}

for race, laps in {
    'Abu Dhabi Grand Prix':      58,
    'Australian Grand Prix':     58,
    'Austrian Grand Prix':       71,
    'Azerbaijan Grand Prix':     51,
    'Bahrain Grand Prix':        57,
    'Belgian Grand Prix':        44,
    'British Grand Prix':        52,
    'Canadian Grand Prix':       70,
    'Chinese Grand Prix':        56,
    'Dutch Grand Prix':          72,
    'Emilia Romagna Grand Prix': 63,
    'Hungarian Grand Prix':      70,
    'Italian Grand Prix':        53,
    'Japanese Grand Prix':       53,
    'Las Vegas Grand Prix':      50,
    'Mexico City Grand Prix':    71,
    'Miami Grand Prix':          57,
    'Monaco Grand Prix':         78,
    'Qatar Grand Prix':          57,
    'Saudi Arabian Grand Prix':  50,
    'Singapore Grand Prix':      62,
    'Spanish Grand Prix':        66,
    'São Paulo Grand Prix':      71,
    'United States Grand Prix':  56,
}.items():
    burn_rate    = fuel_consumption_rates.get(race, 1.75)
    fuel_needed  = round(burn_rate * laps, 1)

    # Add 3kg safety margin (teams always carry a small buffer)
    safety_margin   = 3.0
    optimal_load    = round(min(fuel_needed + safety_margin, 110.0), 1)
    full_load       = 110.0
    excess_fuel     = round(full_load - optimal_load, 1)

    # Time penalty of carrying excess fuel over the whole race
    # Each extra kg costs 0.032s per lap across all laps
    time_penalty_total = round(excess_fuel * FUEL_TIME_EFFECT * laps, 2)

    # Average lap time penalty from carrying full load vs optimal
    avg_lap_penalty = round(excess_fuel * FUEL_TIME_EFFECT, 4)

    fuel_export[race] = {
        'burn_rate_per_lap':    burn_rate,
        'fuel_needed_kg':       fuel_needed,
        'safety_margin_kg':     safety_margin,
        'optimal_load_kg':      optimal_load,
        'full_load_kg':         full_load,
        'excess_fuel_kg':       excess_fuel,
        'time_penalty_total_s': time_penalty_total,
        'avg_lap_penalty_s':    avg_lap_penalty,
        'race_laps':            laps,
    }

    print(f"{race:<32} "
          f"Needed: {fuel_needed:5.1f}kg  "
          f"Optimal: {optimal_load:5.1f}kg  "
          f"Excess: {excess_fuel:4.1f}kg  "
          f"Time penalty: {time_penalty_total:5.1f}s")

with open('/Users/vihaan/Desktop/f1-strategy-ai/Data/fuel_loads.json', 'w') as f:
    json.dump(fuel_export, f, indent=2)

print("\nSaved fuel_loads.json")

Abu Dhabi Grand Prix             Needed: 101.5kg  Optimal: 104.5kg  Excess:  5.5kg  Time penalty:  10.2s
Australian Grand Prix            Needed:  95.7kg  Optimal:  98.7kg  Excess: 11.3kg  Time penalty:  21.0s
Austrian Grand Prix              Needed: 106.5kg  Optimal: 109.5kg  Excess:  0.5kg  Time penalty:   1.1s
Azerbaijan Grand Prix            Needed:  86.7kg  Optimal:  89.7kg  Excess: 20.3kg  Time penalty:  33.1s
Bahrain Grand Prix               Needed: 102.6kg  Optimal: 105.6kg  Excess:  4.4kg  Time penalty:   8.0s
Belgian Grand Prix               Needed:  92.4kg  Optimal:  95.4kg  Excess: 14.6kg  Time penalty:  20.6s
British Grand Prix               Needed:  98.8kg  Optimal: 101.8kg  Excess:  8.2kg  Time penalty:  13.6s
Canadian Grand Prix              Needed: 119.0kg  Optimal: 110.0kg  Excess:  0.0kg  Time penalty:   0.0s
Chinese Grand Prix               Needed: 100.8kg  Optimal: 103.8kg  Excess:  6.2kg  Time penalty:  11.1s
Dutch Grand Prix                 Needed: 111.6kg  Optim